> **GitHub管理版（2026-11-05移管）**
> 既存の実験コードを移管したものです。セル出力は除去しており、今回の移管では実験を再実行していません。入力データ・保存済み結果は従来のDrive等を参照します。実行条件、認証、保存先を確認してから実行してください。元ファイルと用途は [ノートブック一覧](https://github.com/ryy1210/RMT_utils/blob/main/notebooks/README.md) を参照してください。

# Llama・Gemma：両端外れ値を除いたMP bulkKS

**準備済み・本番未実行。CPUで上から「すべてを実行」。GPU・モデル取得・SVD・HF認証は不要。**
保存済み434行列のpre/postスペクトルを使用し、分散は再推定しません。旧ノートブック・元CSV・元スペクトルは変更しません。

## 方法
主解析 `matched_upper` は既存の上端閾値を固定し、下端除外の効果だけを比較します。
固有値は保存された $\lambda=s^2/n$、$p\le n,\gamma=p/n$ です。元の行列形状を維持し、除外後の個数で$\gamma$を再定義しません。

$$a=\sigma^2(1-\sqrt\gamma)^2,\quad b=\sigma^2(1+\sqrt\gamma)^2$$
$$s_\pm=\sigma^2 n^{-2/3}\gamma^{-1/6}(1\pm\sqrt\gamma)^{4/3}$$
既存上端を$U$として $c=(U-b)/s_+$を復元し、$L=\max(0,a-cs_-)$を追加します。
$\lambda<L$と$\lambda>U$を外れ値とし、**境界の値は残す**。$c$は保存済みの補正係数で、推定し直すパラメータではありません。

補助解析 `squeeze_style_c2` は共通幅$\delta=2s_+$で$[\max(0,a-\delta),b+\delta]$を使用します。
[SQUEEZE v1, 式(5)(6)](https://doi.org/10.21203/rs.3.rs-8720929/v1)は両端に同じ幅、係数2と記載。
ただし同節では尺度の具体式・KSの区間内正規化が完全には規定されていないため、**著者実装の厳密再現とは呼びません**。
係数2をTW1の厳密な95%分位点とは扱いません。この補助解析では上端も変わるため、同じ上端の片側KSも計算して比較します。

### 解釈上の条件
- $p=n$では下端はhard edge（0）。下端TWを適用せず、$L=0$として下側除外なし。現行データの正方行列でも正常動作します。
- $0<n-p\le10$は保守的な実装上のガードとして下側除外を無効化。これは普遍的な理論境界ではありません。状態列に記録します。
- 下端式はsoft edgeの一次漸近尺度で、有限標本での誤判定率を校正した検定ではありません。[関連する極値近似の条件](https://arxiv.org/abs/1203.0839)。
- 残った固有値の経験CDFを総質量1に正規化し、元のMP CDFと比較します。今回の区間はMP支持全体を含むため、理論側の条件付きCDFは元のMPと同じです。
- KSはp値ではありません。下端除外によるKS低下は、上端だけを使う現在のLRAの正当性やPPL維持を証明しません。
- KSが小さくなるように閾値を探索しません。preDE・postDE（BEMA）・DE_1（分散1）を同じ手順で比較します。


### 公開コードとの照合（2026-09-26）
確認したcommit: `9aeac9efede635b2b29c88179ca0c8c8b23e2155`、[rmt_lowrank.py](https://github.com/pierredantas/phd-llm-compressor/blob/9aeac9efede635b2b29c88179ca0c8c8b23e2155/src/compression/rmt_lowrank.py#L104-L123)。
`_mp_support` は $L=a+\delta, U=b-\delta$ と **内側** へ切り詰めています。
$\delta=\max(0.15(b-a),1.5p^{-2/3}2^{4/3})$、TW項では`beta=1`に固定。
論文の外側拡張・係数2とは異なります。取得したmainのテキストソースにはKS・bootstrap・BHの実装がありません。
また既定`gavish_donoho`分岐はL/Uを使用せず、特異値の中央値から閾値を決めます。
したがって公開リポジトリから論文のKS結果を厳密再現することは現状できません。

このノートブックの `squeeze_style_c2` は**論文式を参考にした明示的な実装**であり、公開コード再現ではありません。
コードの内側区間に合わせたKSはここでは計算しません。内側に切り詰める場合は理論MPも区間で条件付けするかどうかが別の方法上の選択となり、今の外れ値除外と同一ではないためです。
列標準化したスペクトルは保存されていないので、その前処理まで再現するには重み取得・再SVDが必要です。


In [ ]:
# 初回のみ不足ライブラリをインストール。torch/transformersは不要。
import importlib.util, subprocess, sys
packages = {'numpy':'numpy', 'pandas':'pandas', 'scipy':'scipy', 'matplotlib':'matplotlib'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
from pathlib import Path
IN_COLAB = importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
import json, hashlib, os, platform, datetime
from functools import lru_cache
import numpy as np
import pandas as pd
from scipy import integrate
import scipy
import matplotlib.pyplot as plt
from IPython.display import display

# ローカルのDrive同期フォルダも自動検出。ColabではCPUランタイムを選択。
if IN_COLAB:
    DRIVE_DATA = Path('/content/drive/MyDrive/TUS/hashiguchi/data')
else:
    choices = list((Path.home()/'Library/CloudStorage').glob('GoogleDrive-*/*/TUS/hashiguchi/data'))
    if len(choices) != 1:
        raise RuntimeError('DRIVE_DATAを自分のDriveのdataフォルダに設定してください')
    DRIVE_DATA = choices[0]
SOURCE_TAG = 'bulk-ks-20260925-v1-full'
EXPERIMENT_TAG = 'two-sided-bulk-ks-v1'
EXPECTED_ROWS = {'llama':196, 'gemma':238}
SQUEEZE_C = 2.0
NEAR_HARD_EDGE_GAP = 10
OUTPUT = DRIVE_DATA/'bulk_ks_two_sided'/EXPERIMENT_TAG
PUBLISH_WANDB = False  # 必要ならTrue。KS計算・Drive保存にはAPIキー不要。
AUTO_DISCONNECT = True  # 全保存成功後、Colabの場合のみ解放。
print('CPU analysis; source:', DRIVE_DATA/'bulk_ks', '\noutput:', OUTPUT)


## 計算関数
既存実験と数値を揃えるため、MP積分・KSは既存funcs1.pyの関数本体をそのまま収録しています（20万点・経験CDFの左右を評価）。新規追加部分にはコメントを付けています。

In [ ]:
@lru_cache(maxsize=16)
def _mp_grid(gamma, grid_size=200000):
    # 高速化: 同じ行列形状で繰り返す20万点のMP積分を再利用。
    a = (1.0 - np.sqrt(gamma)) ** 2
    b = (1.0 + np.sqrt(gamma)) ** 2

    # x = a + (b-a)(1-cos(theta))/2
    # gamma=1 の x=0 特異性を避ける
    eps = 1e-6
    theta = np.linspace(eps, np.pi - eps, grid_size)

    x = a + (b - a) * (1.0 - np.cos(theta)) / 2.0
    dx_dtheta = (b - a) * np.sin(theta) / 2.0

    denom = np.maximum(x, np.finfo(float).tiny)

    pdf = np.sqrt(np.maximum((b - x) * (x - a), 0.0)) / (
        2.0 * np.pi * gamma * denom
    )

    integrand = pdf * dx_dtheta

    cdf = integrate.cumulative_trapezoid(integrand, theta, initial=0.0)
    cdf = cdf / cdf[-1]

    # 正確な端点を追加
    cdf_all = np.r_[0.0, cdf, 1.0]
    x_all = np.r_[a, x, b]

    mask = np.isfinite(cdf_all) & np.isfinite(x_all)
    cdf_all = cdf_all[mask]
    x_all = x_all[mask]

    # np.interp 用に重複CDFを除く
    cdf_unique, idx = np.unique(cdf_all, return_index=True)
    x_unique = x_all[idx]

    return x_unique, cdf_unique

def _mp_ks(evals, gamma, sigma2):
    if sigma2 <= 0:
        return 0.0 if np.all(evals == 0) else np.nan
    x, cdf = _mp_grid(gamma)
    theory = np.interp(evals / sigma2, x, cdf, left=0, right=1)
    N = len(evals)
    # 修正: 旧実装の右側のみの比較ではKS距離が過小評価される。
    return float(max(np.max(np.arange(1, N + 1) / N - theory),
                     np.max(theory - np.arange(N) / N)))

In [ ]:
def ks(values, gamma, variance):
    values = np.sort(np.asarray(values, dtype=float))
    return _mp_ks(values, gamma, variance) if len(values) else float('nan')


def bounds(p, n, variance, upper_saved, c_shared=2.0):
    if int(p) != p or int(n) != n or not 0 < p <= n or not np.isfinite(variance) or variance <= 0:
        raise ValueError('Invalid dimensions or variance')
    if not np.isfinite(upper_saved) or not np.isfinite(c_shared) or c_shared < 0:
        raise ValueError('Invalid threshold or correction')
    gamma = p/n
    a, b = variance*(1-np.sqrt(gamma))**2, variance*(1+np.sqrt(gamma))**2
    s_plus = variance*n**(-2/3)*gamma**(-1/6)*(1+np.sqrt(gamma))**(4/3)
    s_minus = variance*n**(-2/3)*gamma**(-1/6)*(1-np.sqrt(gamma))**(4/3)
    c = (upper_saved-b)/s_plus
    if c < -1e-7:
        raise ValueError('Saved upper threshold is inside MP support')
    c = max(c, 0.0)
    status = 'hard_edge' if p == n else ('near_hard_edge_skipped' if n-p <= NEAR_HARD_EDGE_GAP else 'soft_edge')
    # 追加: 下端は反転TWのためMP下端から補正を引く。上端の尺度を流用しない。
    lower = max(0.0, a-c*s_minus) if status == 'soft_edge' else 0.0
    lower_shared = max(0.0, a-c_shared*s_plus) if status == 'soft_edge' else 0.0
    return a, b, s_minus, s_plus, c, status, [
        ('matched_upper', lower, upper_saved),
        ('squeeze_style_c2', lower_shared, b+c_shared*s_plus)]


def analyze_spectrum(values, p, n, variance, upper_saved):
    values = np.sort(np.asarray(values, dtype=float))
    if values.ndim != 1 or len(values) != p or not np.isfinite(values).all() or (values < 0).any():
        raise ValueError('Expected p finite, nonnegative normalized eigenvalues')
    a,b,sm,sp,c,status,intervals = bounds(p,n,variance,upper_saved,SQUEEZE_C)
    records = []
    full = ks(values,p/n,variance)
    for method,lower,upper in intervals:
        low = values < lower
        high = values > upper
        kept = ~(low | high)
        one = values[~high]
        bulk = values[kept]
        assert lower <= a+1e-12 and upper >= b-1e-12
        # 支持全体を含むため、理論MP CDFを切り詰める必要はない。
        rec = dict(method=method, lower_threshold=lower, upper_threshold=upper,
                   mp_lower=a, mp_upper=b, tw_scale_lower=sm, tw_scale_upper=sp,
                   saved_upper_coefficient=c, lower_edge_status=status,
                   n_total=len(values), n_low=int(low.sum()), n_high=int(high.sum()), n_bulk=len(bulk),
                   low_ratio=float(low.mean()), high_ratio=float(high.mean()), bulk_ratio=float(kept.mean()),
                   fullKS=full, upper_only_bulkKS=ks(one,p/n,variance),
                   two_sided_bulkKS=ks(bulk,p/n,variance),
                   status='ok' if len(bulk) else 'empty_bulk')
        rec['delta_two_minus_upper'] = rec['two_sided_bulkKS']-rec['upper_only_bulkKS']
        records.append(rec)
    return records


## 小さな動作確認
人工的な固有値配列だけを使い、両側除外・境界・空バルク・正方行列の処理を確認します。研究結果ではありません。

In [ ]:
# 両端の外れ値を追加した人工スペクトル。行列SVDは行わない。
p,n,var=100,400,1.0
g=p/n; x,F=_mp_grid(g)
u=var*(1+np.sqrt(g))**2+0.45*var*n**(-2/3)*g**(-1/6)*(1+np.sqrt(g))**(4/3)
v=np.interp((np.arange(p)+0.5)/p,F,x); v[0]=0; v[-1]=u+1
r=analyze_spectrum(v,p,n,var,u)[0]
assert (r['n_low'],r['n_high'],r['n_bulk'])==(1,1,98)
# 再正規化で距離が必ず下がるとは限らないため、改善そのものをテスト条件にしない。
assert np.isfinite(r['two_sided_bulkKS'])
assert r['n_total']==r['n_low']+r['n_high']+r['n_bulk']
square=analyze_spectrum(np.linspace(0,4,100),100,100,1,4.1)[0]
assert square['lower_edge_status']=='hard_edge' and square['n_low']==0
assert square['two_sided_bulkKS']==square['upper_only_bulkKS']
empty=analyze_spectrum(np.zeros(100),100,400,1,u)[0]
assert empty['status']=='empty_bulk' and np.isnan(empty['two_sided_bulkKS'])
print('Synthetic checks passed. No model or SVD used.')


## 入力確認と再開準備
元CSVとmetadataを確認し、設定や入力が異なる結果への追記を拒否します。計算途中の結果は行列ごとに保存します。スペクトルのSHA-256も記録し、再開時に同一性を確認します。

In [ ]:
def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def atomic_csv(frame,path):
    temp=path.with_suffix('.tmp.csv')
    frame.to_csv(temp,index=False)
    temp.replace(path)

sources={}; source_ids={}
for model,expected in EXPECTED_ROWS.items():
    directory=DRIVE_DATA/'bulk_ks'/model/SOURCE_TAG
    frame=pd.read_csv(directory/'bulk_ks.csv')
    meta=json.loads((directory/'metadata.json').read_text())
    if len(frame)!=expected or frame.name.duplicated().any() or set(frame.name)!=set(meta['config']['layers']):
        raise ValueError(f'{model}: input coverage mismatch')
    if not np.allclose(frame.gamma,frame.p/frame.n):
        raise ValueError('Aspect ratio mismatch')
    sources[model]=(directory,frame)
    source_ids[model]={'csv_sha256':sha(directory/'bulk_ks.csv'), 'metadata_sha256':sha(directory/'metadata.json'),
                       'source_run': (directory/'wandb_run_url.txt').read_text().strip()}
# 関数バイトコードから実装の指紋を生成。Python版の違いも別設定として扱う。
import marshal
functions=[_mp_grid.__wrapped__,_mp_ks,ks,bounds,analyze_spectrum]
import types
def stable_code(code):
    # co_filename/co_firstlinenoはJupyterのセル実行番号で変わるため正規化。
    consts=tuple(stable_code(c) if isinstance(c,types.CodeType) else c for c in code.co_consts)
    return code.replace(co_filename='<stable>',co_firstlineno=1,co_consts=consts)
implementation=hashlib.sha256(b''.join(marshal.dumps(stable_code(f.__code__)) for f in functions)).hexdigest()
config={'sources':source_ids,'source_tag':SOURCE_TAG,'squeeze_c':SQUEEZE_C,
        'near_hard_edge_gap':NEAR_HARD_EDGE_GAP,'implementation_sha256':implementation,
        'python':platform.python_version(),'numpy':np.__version__,'scipy':scipy.__version__,
        'pandas':pd.__version__,'scale':'lambda=s^2/n','primary':'matched_upper'}
identity=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()
OUTPUT.mkdir(parents=True,exist_ok=True)
manifest=OUTPUT/'metadata.json'
if manifest.exists():
    if json.loads(manifest.read_text())['identity']!=identity:
        raise RuntimeError('設定/入力/コードが異なります。EXPERIMENT_TAGを変更してください')
else:
    if any(OUTPUT.iterdir()): raise RuntimeError('metadataなしの既存出力があります。別TAGを使用してください')
    manifest.write_text(json.dumps({'identity':identity,'config':config},ensure_ascii=False,indent=2))
result_path=OUTPUT/'two_sided_bulk_ks.csv'
records=pd.read_csv(result_path).to_dict('records') if result_path.exists() else []
keys=[(r['model'],r['name'],r['condition'],r['method']) for r in records]
if len(keys)!=len(set(keys)): raise ValueError('Duplicate checkpoint rows')
expected_keys={(m,name,c,method) for m,(_,f) in sources.items() for name in f.name
               for c in ['preDE','postDE','DE_1'] for method in ['matched_upper','squeeze_style_c2']}
if not set(keys)<=expected_keys: raise ValueError('Unexpected checkpoint rows')
print('Input: 434 matrices; existing result rows:',len(records))


## 実データの計算
1行列につき3条件×2手法を保存します。`matched_upper`の全体KS・片側bulkKSが元CSVと一致することを毎回確認し、誤った正規化・入力の取り違えを検出します。

In [ ]:
import io
completed={(r['model'],r['name']) for r in records}
for model,(directory,frame) in sources.items():
    for i,(_,row) in enumerate(frame.iterrows(),1):
        spectrum_path=(directory/row.spectrum_file).resolve()
        if not spectrum_path.is_relative_to(directory.resolve()): raise ValueError('Invalid spectrum path')
        # 各npzは一度だけ読み、内容とハッシュを同じバイト列から取得。
        payload=spectrum_path.read_bytes()
        spectrum_sha=hashlib.sha256(payload).hexdigest()
        if (model,row['name']) in completed:
            previous=[r for r in records if r['model']==model and r['name']==row['name']]
            if len(previous)!=6 or any(r['spectrum_sha256']!=spectrum_sha for r in previous):
                raise ValueError('Incomplete or changed spectrum checkpoint')
            continue
        with np.load(io.BytesIO(payload),allow_pickle=False) as data:
            p,n=int(data['p']),int(data['n'])
            if p!=row.p or n!=row.n or str(data['layer_name'].item())!=row['name']:
                raise ValueError('Spectrum identity mismatch')
            pending=[]
            for condition,spectrum_key,full_col in [('preDE','pre','KS_preDE'),('postDE','post','KS_postDE'),('DE_1','post','KS_postDE_1')]:
                variance=float(row['sigma2_'+condition]); upper=float(row['threshold_'+condition])
                result=analyze_spectrum(data[spectrum_key],p,n,variance,upper)
                main=result[0]
                if not np.isclose(main['upper_only_bulkKS'],row['bulkKS_'+condition],rtol=0,atol=2e-7):
                    raise ValueError(f'Previous bulkKS mismatch: {model} {row["name"]} {condition}')
                if not np.isclose(main['fullKS'],row[full_col],rtol=0,atol=2e-7):
                    raise ValueError('Previous fullKS mismatch')
                for r in result:
                    r.update(model=model,name=row['name'],condition=condition,p=p,n=n,gamma=p/n,sigma2=variance,
                             module_type=row.module_type,block_index=int(row.block_index),spectrum_sha256=spectrum_sha,
                             fullKS_previous=row[full_col],bulkKS_previous=row['bulkKS_'+condition],
                             fullKS_historical=row[full_col+'_saved'])
                    pending.append(r)
        records.extend(pending)
        atomic_csv(pd.DataFrame(records),result_path)
        print(f'{model} [{i}/{len(frame)}] {row["name"]}: saved',flush=True)
results=pd.DataFrame(records)
assert len(results)==434*3*2
assert (results.n_total==results.n_low+results.n_high+results.n_bulk).all()
assert (results.n_bulk>=0).all()
COMPUTATION_COMPLETE=True
print('Complete:',len(results),'rows; empty bulks:',int((results.status=='empty_bulk').sum()))


## 集計と図
空のバルクを適合扱いしないよう、総行列数・KSが定義できた数を併記します。0.05は距離の目安で、検定の有意水準ではありません。

In [ ]:
def summarize(frame,groups):
    return frame.groupby(groups).agg(matrices=('name','size'),valid=('two_sided_bulkKS','count'),
        upper_only_median=('upper_only_bulkKS','median'),two_sided_median=('two_sided_bulkKS','median'),
        paired_delta_median=('delta_two_minus_upper','median'),lower_removed_median=('low_ratio','median'),
        bulk_ratio_median=('bulk_ratio','median'),
        improved=('delta_two_minus_upper',lambda x:int((x<0).sum())),
        below_005=('two_sided_bulkKS',lambda x:int((x<.05).sum()))).reset_index()
summary=summarize(results,['model','method','condition'])
modules=summarize(results,['model','method','condition','module_type'])
atomic_csv(summary,OUTPUT/'summary.csv');atomic_csv(modules,OUTPUT/'module_summary.csv')
paired=results.pivot(index=['model','name','method'],columns='condition',values='two_sided_bulkKS').reset_index()
paired['delta_post_minus_pre']=paired.postDE-paired.preDE
atomic_csv(paired,OUTPUT/'paired_de.csv')
display(summary)
display(paired.groupby(['model','method']).agg(valid_pairs=('delta_post_minus_pre','count'),
    improved_by_DE=('delta_post_minus_pre',lambda x:int((x<0).sum())),
    paired_delta_median=('delta_post_minus_pre','median')))
plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False})
for method in ['matched_upper','squeeze_style_c2']:
    fig,axes=plt.subplots(2,3,figsize=(13,8),layout='constrained')
    for i,model in enumerate(['llama','gemma']):
        for j,condition in enumerate(['preDE','postDE','DE_1']):
            d=results[(results.model==model)&(results.method==method)&(results.condition==condition)]
            ax=axes[i,j];points=ax.scatter(d.upper_only_bulkKS,d.two_sided_bulkKS,c=d.low_ratio,vmin=0,vmax=1,cmap='viridis',s=16)
            ax.plot([0,1],[0,1],'--',color='gray',lw=1)
            ax.set(xlim=(0,1),ylim=(0,1),xlabel='Upper-only bulk KS',ylabel='Two-sided bulk KS',title=f'{model} / {condition} (n={len(d)})')
    fig.colorbar(points,ax=axes.ravel().tolist(),label='Lower-outlier fraction')
    fig.suptitle(method+' — below diagonal: KS decreases after lower trimming')
    fig.savefig(OUTPUT/(method+'_comparison.png'),dpi=160)
    plt.show()
fig,axes=plt.subplots(1,2,figsize=(11,4),layout='constrained')
for ax,model in zip(axes,['llama','gemma']):
    d=paired[(paired.model==model)&(paired.method=='matched_upper')]
    ax.scatter(d.preDE,d.postDE,s=18,color='#2864a0')
    ax.plot([0,1],[0,1],'--',color='gray')
    ax.set(xlim=(0,1),ylim=(0,1),xlabel='Two-sided KS before DE',ylabel='Two-sided KS after DE',title=model)
fig.savefig(OUTPUT/'two_sided_de_comparison.png',dpi=160);plt.show()


## W&B（任意）と完了
Driveには自動保存します。W&Bは設定でTrueにした場合だけ新規runへ送信します。`WANDB_API_KEY`をos.environに明示設定してから実行できます。HF_TOKENは不要です。

In [ ]:
PUBLISH_COMPLETE = not PUBLISH_WANDB
if PUBLISH_WANDB:
    if importlib.util.find_spec('wandb') is None:
        subprocess.run([sys.executable,'-m','pip','install','wandb'],check=True)
    import wandb
    if not os.environ.get('WANDB_API_KEY'):
        raise RuntimeError('WANDB_API_KEYをos.environへ設定してください。結果はDriveに保存済みです。')
    receipt=OUTPUT/'wandb_run_url.txt'
    if receipt.exists():
        print('Already published:',receipt.read_text().strip())
    else:
        wandb.login(key=os.environ['WANDB_API_KEY'])
        with wandb.init(entity='ryoya-zushi1210-tokyo-university-of-science',project='LlaMa-RMT-Analysis',
                        job_type='two-sided-bulk-ks',name=EXPERIMENT_TAG,config=config) as run:
            run.log({'two_sided_bulk_ks':wandb.Table(dataframe=results),'summary':wandb.Table(dataframe=summary)})
            artifact=wandb.Artifact('llama-gemma-two-sided-bulk-ks',type='diagnostics')
            artifact.add_dir(str(OUTPUT));run.log_artifact(artifact)
            run_url=run.url
        receipt.write_text(run_url+'\n')
    PUBLISH_COMPLETE=True
print('Results saved:',OUTPUT)


## 解釈のチェック
1. 下側除外による改善幅を、除外率とともに見る。
2. 正方行列では主解析の片側KSと両側KSが一致することを確認する。
3. 下側を除くと小さくなる場合、現在の上側だけのLRAで「ノイズ」とした集合にはMPに合わない小固有値も含まれていた可能性がある。
4. 前処理はDEであり、列標準化を使うSQUEEZEそのものの再現実験ではない。
5. MP適合と低ランク近似後の性能は別に検証する。

In [ ]:
# 成功時だけDrive同期・Colab解放。ローカルCPUの場合は何もしない。
assert COMPUTATION_COMPLETE and PUBLISH_COMPLETE
if IN_COLAB and AUTO_DISCONNECT:
    from google.colab import drive, runtime
    drive.flush_and_unmount()
    print('Drive synchronized; releasing Colab runtime.')
    runtime.unassign()
